<a href="https://colab.research.google.com/github/oumaklaus/Deathon-ML-Internship/blob/main/California_Housing_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/oumaklaus/Deathon-ML-Internship/blob/main/California_Housing_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


## 1. Data Loading


In [ ]:
import urllib.request

base_url = "https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv"
housing_file = "housing.csv"
urllib.request.urlretrieve(base_url, housing_file)


In [ ]:
!ls


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import (mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score)

RANDOM_STATE = 42
sns.set_theme(style = "whitegrid")


In [ ]:
df = pd.read_csv("housing.csv")


# 2. EDA


In [ ]:
df.shape


In [ ]:
df.head()


In [ ]:
df.tail()


In [ ]:
df.info()


In [ ]:
df.describe()


In [ ]:
df.isnull().sum()


In [ ]:
df['ocean_proximity'].value_counts()


In [ ]:
sns.histplot(data=df, x = "median_house_value", bins=40)
plt.show()


In [ ]:
sns.histplot(data=df, x = "median_income", bins=40)
plt.show()


In [ ]:
sns.histplot(data=df, x = "housing_median_age", bins=40)
plt.show()


In [ ]:
sns.boxplot(data=df, x = "ocean_proximity", y = "median_house_value")
plt.xticks(rotation=45)
plt.show()


In [ ]:
# only numeric columns -> drop the categorical one for a numeric correlation matrix
sns.heatmap(df.drop(columns='ocean_proximity').corr(), annot=True, fmt=".2f")
plt.show()


In [ ]:
sns.scatterplot(data=df, x = "median_income", y = "median_house_value", alpha=0.4)
plt.show()


## 3. Handling Missing Values


In [ ]:
# total_bedrooms has some missing values -> fill with the median
df = df.copy()
df['total_bedrooms'] = df['total_bedrooms'].fillna(df['total_bedrooms'].median())
print(df.isnull().sum().sum(), "missing values remain")


## 4. Dataset Splitting


In [ ]:
X = df.drop(columns = "median_house_value")
y = df["median_house_value"]


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = RANDOM_STATE)


In [ ]:
print("X Train: ", X_train.shape)


In [ ]:
print("X Test: ", X_test.shape)


## 5. Encoding & Feature Scaling


In [ ]:
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = ["ocean_proximity"]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])


In [ ]:
X_train_s = preprocessor.fit_transform(X_train)
X_test_s = preprocessor.transform(X_test)


## 6. Model Training


In [ ]:
lin_reg = LinearRegression()
tree = DecisionTreeRegressor(max_depth=5, random_state=RANDOM_STATE)

lin_reg.fit(X_train_s, y_train)
tree.fit(X_train_s, y_train)

pred_lin = lin_reg.predict(X_test_s)
pred_tree = tree.predict(X_test_s)


## 7. Model Evaluation


In [ ]:
def report(name, y_true, y_pred):
    print(name)
    print(f"  MAE : {mean_absolute_error(y_true, y_pred):,.2f}")
    print(f"  RMSE: {root_mean_squared_error(y_true, y_pred):,.2f}")
    print(f"  R^2 : {r2_score(y_true, y_pred):.4f}")
    print()

report("Linear Regression", y_test, pred_lin)
report("Decision Tree", y_test, pred_tree)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (name, y_pred) in zip(axes, [("Linear Regression", pred_lin), ("Decision Tree", pred_tree)]):
    ax.scatter(y_test, y_pred, alpha=0.4)
    lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    ax.plot(lims, lims, 'r--', label="perfect prediction")
    ax.set_xlabel("True value")
    ax.set_ylabel("Predicted value")
    ax.set_title(f"{name} (R^2 = {r2_score(y_test, y_pred):.3f})")
    ax.legend()
plt.tight_layout()
plt.show()


## 8. Feature Importance (Decision Tree)


In [ ]:
feature_names = num_cols + list(preprocessor.named_transformers_['cat'].get_feature_names_out(['ocean_proximity']))
imp = pd.Series(tree.feature_importances_, index=feature_names).sort_values()
imp.plot(kind="barh", title="Decision Tree feature importance", figsize=(8, 6))
plt.show()


## 9. Cross-Validation


In [ ]:
cv_scores = cross_val_score(tree, X_train_s, y_train, cv=5, scoring="r2")
print(f"Decision Tree CV R^2: {cv_scores}")
print(f"Mean CV R^2         : {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
